# Lesson 10 — Introduction to Phaser

## 1. Context & Goal

This project implements a **Digit Symbol Substitution Test (DSST)** task. DSST is a neuropsychological test sensitive to brain damage, dementia, age, and depression. Participants are shown a key containing paired graphics (like numbers and abstract symbols) and are tasked with matching other graphical symbols according to the key as quickly as possible.

We are building this using **Phaser**, an HTML5 game framework. 

**Why use a game framework for an experiment?**
Behavioral experiments require highly precise timing, fluid rendering of graphics, structured sequences (trials and blocks), and responsive user inputs. Phaser handles all of this elegantly. Instead of dealing with clunky standard HTML/CSS for animations or complex timing logic from scratch, we use Phaser's engine to orchestrate the entire experiment—rendering symbols, running countdowns, recording how fast users click, and seamlessly transitioning between practice rounds and the real tests.

## 1.5 Essential Keywords

When traversing Phaser code, you will constantly see these specific JavaScript keywords. Understanding them is critical:

### `extends` (Inheritance)
In JavaScript, `extends` creates a **child class** that inherits everything from a **parent class**. 

When we write `class Boot extends Phaser.Scene`, we are saying: *"Create a new class named `Boot`, and automatically give it all the hundreds of built-in features that a standard `Phaser.Scene` has."* You don't have to write the complex engine logic yourself; you inherit it for free and only write the specific logic for your scene.

### `super()` (Calling the Parent Constructor)
When you extend a class, you almost always use `super()` inside your `constructor()`. `super()` literally runs the `constructor` function of the parent class. 

By calling `super({ key: "boot" })`, you are telling JavaScript: *"Before I set up my specific Boot scene variables, please execute the main `Phaser.Scene` setup first so my underlying foundation is ready to go."*

### `this` (Current Instance)
You already saw this in Lesson 8, but it's used *everywhere* in Phaser. `this` refers to the **current instance** of the object executing the code.
- **In a Scene:** `this.add.image()` means "Hey scene, use *your* add method to place an image."
- **In a custom class:** `this.scene = scene;` means "Save the scene I was given into *my specific object's* memory so I can use it later."

### `static` (Class-Level Properties)
When a property or method is marked as `static`, it means it belongs to the **Blueprint/Class itself**, not to individual instances created by `new`.

```javascript
export default class Boot extends Phaser.Scene {
    static primaryColor = "#1a1a1a";
}

// You don't need 'new Boot()' to use it! You just call it directly:
console.log(Boot.primaryColor);
```
In this task, `Boot` uses `static` for things like `Boot.canvasWidth` so that *other* scenes can easily read those values without having to create a brand new Boot scene object.

### `export default` and `import`
This is how JavaScript shares code across different files.
- **`export default class Boot { ... }`**: Means "If another file asks for this file, give them this `Boot` class."
- **`import Boot from "./scenes/boot.js";`**: (seen at the top of `main.js`) Means "Go into that file and bring in whatever it exported so I can use it here."

### `new` (Instantiation)
As covered in Lesson 8, `new` is the trigger word that tells JavaScript to actually **build an object** using a class blueprint.

```javascript
// A blueprint sitting doing nothing
class Countdown { ... } 

// Actually building the timer and putting it on the screen!
let timer = new Countdown(this, 100, 100); 
```

## 2. Phaser’s Class Model (What to Know)

As you learned in Lesson 8, JavaScript uses **classes** as blueprints to create objects. Phaser completely revolves around this object-oriented, class-based architecture. Understanding these four core classes is the key to understanding how our experiment works:

1. **`Phaser.Game`**: 
   - Think of this as the **Command Center**. It bootstraps the engine, creates the HTML `<canvas>` element (the black box where everything is drawn), reads your display settings, and dictates which screen should load first. Everything runs under the umbrella of a single `Phaser.Game` instance.
2. **`Phaser.Scene`**: 
   - Think of scenes as **Screens** or **Chapters** in a movie. A game has a main menu scene, a playing scene, a game over scene. Our experiment has a Practice scene, a Run scene, and an Instructions scene. Each scene manages its own lifecycle using default methods: `preload()` (load images), `create()` (place images on screen), and `update()` (run logic 60 times a second).
3. **`Phaser.GameObjects.Container`**:
   - Think of this as a **Folder**. Sometimes you want a button that has a background image, a border, and text. Instead of moving all three manually, you put them inside a `Container`. Then, you scale, move, or hide the Container, and everything inside it behaves as a single unit. 
4. **`Phaser.Events.EventEmitter`**: 
   - Think of this as a **Radio Station**. Different parts of code can tune in. When a trial finishes, a class can `emit` a "trial_complete" event. The Scene can be listening for that event and know it is time to load the next trial or show a "Good job!" screen. This prevents spaghetti code and keeps things organized.

## 3. How This Task Uses Phaser Classes

Let's look at how these concepts apply to the actual code in the `wecare_dsst-main/src/` folder.

* **`src/main.js`**: 
  - This is the entry point. It builds the massive `config` dictionary telling Phaser the width, height (`360 * assetSize` by `640 * assetSize`), physics settings, and background color (`Boot.primaryColor`).
  - It creates `new Phaser.Game(config)` to launch the framework. 
  - It also registers every Scene the game will ever need: `[Boot, Border, DialogBox, Start, Baseline, Practice, Run]`.

* **`src/scenes/boot.js`**: `class Boot extends Phaser.Scene`
  - This is the absolute very first screen because it's first in the `scene` array inside `main.js`.
  - It creates static canvas properties and reads CSS variables via `getComputedStyle()`.
  - It preloads "atlases" (giant images containing all the symbols needed so the browser only has to download one big image instead of fifty small ones).
  - Once loading is done, it starts the `Start` scene.

* **`src/scenes/start.js`**: `class Start extends Phaser.Scene`
  - Displays the opening screen. 
  - Uses a custom `Button` object instance to capture participant clicks before transitioning to the `DialogBox` scene for instructions.

* **`src/scenes/baseline.js`, `practice.js`, `run.js`**: `extends Phaser.Scene`
  - These are the core engines for the participant's task.
  - Rather than filling the `create()` method with a thousand lines of trial code, they cleanly construct helper classes: `TrialHandler`, `GameUI`, `Countdown`, and `GameInstructions`.
  - They use Phaser's event systems safely sequence the task logic (e.g., when the `Countdown` finishes, start the `TrialHandler`).

* **`src/scenes/dialogbox.js`**: 
  - An interstitial scene. It pauses the experiment, shows large blocks of instructional text (using our custom `Message` class) and handles transitions, ensuring participants know what to do next.

* **`src/scenes/border.js`**:
  - A persistent background scene. In Phaser, multiple scenes can run simultaneously. This scene draws the fixed visual border and stays active underneath the other scenes, providing visual consistency.

## 4. Custom Classes (Not Phaser, but using Phaser)

Look inside the `src/gameobjects/` folder. You will find files like `GameUI`, `TrialHandler`, `Countdown`, `GameInstructions`, and `Message`. 

If you open them, you will notice they **do not** say `extends Phaser...`. They are plain JavaScript classes. Why?

Phaser Scenes are heavy. You don't want a scene for every tiny logical component. Instead, we write standard JavaScript classes but we **pass the active Scene as an argument to the constructor**. 

```javascript
class Countdown {
   // We pass the "scene" in so Countdown can tap into Phaser's powers!
   constructor(scene, x, y) {
      this.scene = scene;

      // Because we have the scene, we can ask Phaser to draw text for us!
      this.textObject = this.scene.add.text(x, y, 'Ready?');
   }
}
```

Because they save a reference to `this.scene`, they are allowed to use Phaser APIs. They can draw shapes, start timers (`scene.time.addEvent`), animate tweens (`scene.tweens.add`), and emit events cleanly without carrying the heavy baggage of a full `Phaser.Scene`.

## 5. Concrete Code Snippets

Let's look at exactly how some of these concepts appear in the actual source code.

In [ ]:
// 1. Starting the Game Engine (from main.js)
const config = {
  type: Phaser.AUTO,
  width: 360 * assetSize,
  height: 640 * assetSize,
  backgroundColor: Boot.primaryColor,
  physics: {
    default: "arcade",
    arcade: {
      gravity: { y: 350 }
    },
  },
  // Registering all scenes here gives Phaser the map of the game
  scene: [Boot, Border, DialogBox, Start, Baseline, Practice, Run],
};

// This single line turns on everything
const game = new Phaser.Game(config);

In [ ]:
// 2. Creating a Scene (from boot.js)
export default class Boot extends Phaser.Scene {
    // The super call registers the unique key 'boot' for this scene
    constructor() {
        super({ key: "boot"})
    }
    // ... preload(), create(), update() will go here
}

In [ ]:
// 3. Extending a Phaser Container (from button.js)
// We extend Container so we can stack a graphic and text and bind them as one object.
export default class Button extends Phaser.GameObjects.Container {
    constructor(scene, x, y, texture, frameUp, frameDown, labelText = "") {
        super(scene, x, y);
        this.texture = texture;
        
        // This tells the scene to actually draw our newly constructed container
        scene.add.existing(this);  
    }
}

In [ ]:
// 4. Using EventEmitter for Custom Logic (from trialHandler.js)
export default class TrialHandler {
    constructor({ scene, numTrials, /* ... */ }) {
        this.scene = scene;
        
        // We equip this plain class with a Phaser Radio Station!
        this.event = new Phaser.Events.EventEmitter();
    }
    
    finish() {
        // Now it can broadcast its completion, and scenes can listen to it.
        this.event.emit("complete");
    }
}

## 6. Scene Flow Summary

How does a participant travel through the experiment code?

1. **Boot**: Loads all required visual assets (images, UI components) and calculates display sizes. Automatically launches `Border` and transitions to `Start`.
2. **Start**: The landing page screen registering the user is ready.
3. **DialogBox**: Displays introductory instructions in an interstitial box.
4. **Baseline**: Registers base reaction times before task manipulation.
5. **Practice**: Warm-up trials. This scene builds a `TrialHandler` which loops through fake trials to teach the participant the controls.
6. **DialogBox**: Shows more instructions ("Get ready for the real thing").
7. **Run**: The core recorded participant trials tracking success and reaction times.
8. **Start**: Cycles back to conclusion.

## 7. Learning Check / Mini Exercise

To really understand how the parts connect, try executing these short explorations in the codebase:

1. **Investigate Scene Registration:**
   - Open `src/main.js`. Look at the `scene: []` array inside the config object. Look at what happens if you swapped `Start` and `Boot`. (Don't do it, but understand why `Boot` must be first!).
2. **Tweak the Practice Trials:**
   - Open `src/scenes/practice.js`.
   - Find where `new TrialHandler(...)` is constructed.
   - Look for the `numTrials` property being passed in. Try changing it from whatever value it is to something much smaller or larger to see how the object configurations manage the flow.
3. **Trace a Custom Class:**
   - Open `src/gameobjects/countdown.js`.
   - Notice how it accepts `scene` in the `constructor(scene, ...)`? Find a line of code where it utilizes that `scene` to interface with Phaser, such as triggering an event or drawing text via `this.scene.add.text(...)`.